In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
from delta.tables import DeltaTable
dbutils.widgets.text("increm_flag","0")
increm_flag = dbutils.widgets.get("increm_flag")


In [0]:
df_src = spark.read.table("sales.silver.customers")
df_src = df_src.dropDuplicates(subset = ["customer_id"])
df_src.display()

In [0]:
if spark.catalog.tableExists("sales.gold.customers_new"):
    df_sink = spark.sql("select * from sales.gold.customers_new")
else:
    df_sink = spark.sql("select *,0 dim_customer_key,0 create_date,0 update_date from sales.silver.customers where 1 =0")

df_sink.display()


In [0]:
df_sink = df_sink.withColumnRenamed("dim_customer_key","old_dim_customer_key") \
    .withColumnRenamed("create_date","old_create_date") \
    .withColumnRenamed("update_date","old_update_date")

In [0]:
df_join = df_src.join(df_sink,df_src.customer_id == df_sink.customer_id,"left").select(df_src["*"],df_sink.old_dim_customer_key,df_sink.old_create_date,df_sink.old_update_date)
df_join.display()

In [0]:
df_old =df_join.where(df_join.old_dim_customer_key.isNotNull())
df_new = df_join.where(df_join.old_dim_customer_key.isNull())
df_old.display()
df_new.display()


In [0]:
df_old = df_old.drop("old_update_date")
df_old = df_old.withColumnRenamed("old_create_date","create_date")
df_old = df_old.withColumnRenamed("old_dim_customer_key","dim_customer_key")
df_old = df_old.withColumnRenamed("old_update_date","update_date")
df_old = df_old.withColumn("create_date",F.to_timestamp(F.col("create_date")))
df_old = df_old.withColumn("update_date",F.current_timestamp()).drop("_rescued_data")
df_old.display()

In [0]:
df_new = df_new.drop("old_dim_customer_key","old_create_date","old_update_date")
df_new = df_new.withColumn("dim_customer_key",F.monotonically_increasing_id() +F.lit(1)).drop("_rescued_data")
df_new = df_new.withColumn("create_date",F.current_timestamp())
df_new = df_new.withColumn("update_date",F.current_timestamp())

df_new.display()


In [0]:
if (increm_flag == "0"):
    max_value = 0
else:
    if spark.catalog.tableExists("sales.gold.customers_new"):
        df  = spark.sql("select max(dim_customer_key) from sales.gold.customers_new")
        max_value = df.collect()[0][0]
max_value

In [0]:
df_new = df_new.withColumn("dim_customer_key",F.lit(max_value) + F.col("dim_customer_key"))
df_new.display()


In [0]:
df_final = df_new.unionByName(df_old)
df_final.display()

In [0]:
if spark.catalog.tableExists("sales.gold.customers_new"):
    delta_table = DeltaTable.forName(spark,"sales.gold.customers_new")
    delta_table.alias("t").merge(df_final.alias("s"),"t.dim_customer_key = s.dim_customer_key").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.format("delta").mode("append").saveAsTable("sales.gold.customers_new")

In [0]:
%sql
select * from sales.gold.customers_new;